## 依赖导入

本 notebook 聚焦线性规划建模与求解：`cvxpy` 建立并求解（整数）线性规划问题，`numpy` 及 `from numpy import array` 提供向量/矩阵运算，`pandas` 读写数据与结果，`matplotlib.pyplot` 绘制收益-风险等曲线。

In [ ]:
import cvxpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from numpy import array

## 全局绘图配置

设置全局 `plt.rc`：启用 TeX 渲染数学公式、字号 15、中文字体 SimHei，保证 2.12 节收益随风险变化的曲线等图的中文与公式标签正常显示。

In [ ]:
plt.rc("text", usetex=True)
plt.rc("font", size=15)
plt.rc("font", family="SimHei")

## 2.2 三变量线性规划（利润最大化）

目标系数 `c=[70,50,60]`、约束矩阵 `a`、右端项 `b=[150,160,200]`（源程序 ex4_2.py）。用 `cvxpy` 定义 3 个非负决策变量（`pos=True`），目标 `cp.Maximize(c @ x)`，约束 `a @ x <= b`，指定 `solver="GLPK_MI"` 求解并打印最优解与最优值。

In [ ]:
c = array([70, 50, 60])
a = array([[2, 4, 3], [3, 1, 5], [7, 3, 5]])
b = array([150, 160, 200])
x = cp.Variable(3, pos=True)
obj = cp.Maximize(c @ x)
cons = [a @ x <= b]
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优解为：", x.value)
print("最优值为：", prob.value)

## 2.3 多期投资线性规划

`x` 为 $5\times 4$ 非负决策矩阵，表示各期资金在不同项目间的分配（源程序 ex4_3.py）；目标最大化末期复合收益 $1.15x_{3,0}+1.40x_{1,2}+1.25x_{2,1}+1.06x_{4,3}$，各期资金守恒（含 1.06、1.15 等增长率）构成等式约束，另有两个项目的额度上限约束。用 `GLPK_MI` 求解并打印最优值与最优解。

In [ ]:
x = cp.Variable((5, 4), pos=True)
obj = cp.Maximize(1.15 * x[3, 0] + 1.40 * x[1, 2] + 1.25 * x[2, 1] + 1.06 * x[4, 3])
cons = [
    x[0, 0] + x[0, 3] == 100000,
    x[1, 0] + x[1, 2] + x[1, 3] == 1.06 * x[0, 3],
    x[2, 0] + x[2, 1] + x[2, 3] == 1.15 * x[0, 0] + 1.06 * x[1, 3],
    x[3, 0] + x[3, 3] == 1.15 * x[1, 0] + 1.06 * x[2, 3],
    x[4, 3] == 1.15 * x[2, 0] + 1.06 * x[3, 3],
    x[2, 1] <= 40000,
    x[1, 2] <= 30000,
]
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优值为：", prob.value)
print("最优解为：", x.value)

## 2.4 排班/采购整数规划（4×4 决策矩阵）

`x` 为 $4\times 4$ 非负决策矩阵（源程序 ex4_4.py）。目标最小化各列成本加权和 $2800\sum x_{:,0}+4500\sum x_{:,1}+6000\sum x_{:,2}+7300\,x_{0,3}$；约束给出若干累计需求下限。为忠实保留原建模，切片求和 `sum(...)` 与全部约束表达式均按原样保留。用 `GLPK_MI` 求解并打印最优值与解矩阵。

In [ ]:
x = cp.Variable((4, 4), pos=True)
obj = cp.Minimize(
    2800 * sum(x[:, 0]) + 4500 * sum(x[:3, 1]) + 6000 * sum(x[:2, 2]) + 7300 * x[0, 3]
)
cons = [
    sum(x[0, :]) >= 15,
    sum(x[0, 1:]) + sum(x[2, :3]) >= 10,
    sum(x[0, 2:4]) + sum(x[1, 1:3]) + sum(x[2, :2]) >= 20,
    x[0, 3] + x[1, 2] + x[2, 1] + [3, 0] >= 12,
]
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优值为：", prob.value)
print("最优解为：\n", x.value)

## 2.5 运输问题的整数规划（从文本读入）

从 `data/4_5_1.txt` 分三段读入：单位成本矩阵 `c`（前 6 行、前 8 列）、供应上限向量 `e`（前 6 行末列）、需求向量 `d`（第 6 行之后）（源程序 ex4_5_1.py）。`x` 为 $6\times 8$ 非负整数变量，目标最小化 `cp.sum(cp.multiply(c, x))`；约束为各列销量等于需求 `d`、各行发货不超过上限 `e`。用 `GLPK_MI` 求解，打印最优值与解矩阵，并把解写入 `data/4_5_2.xlsx`。

In [ ]:
c = np.genfromtxt("data/4_5_1.txt", dtype=float, max_rows=6, usecols=range(8))
e = np.genfromtxt("data/4_5_1.txt", dtype=float, max_rows=6, usecols=8)
d = np.genfromtxt("data/4_5_1.txt", dtype=float, skip_header=6)
x = cp.Variable((6, 8), pos=True)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)))
con = [cp.sum(x, axis=0) == d, cp.sum(x, axis=1) <= e]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
xd = pd.DataFrame(x.value)
xd.to_excel("data/4_5_2.xlsx")

## 2.5 运输问题的整数规划（从 Excel 读入）

同一运输问题的第二种数据读取方式（源程序 data4_5_2.py）：从 `data/4_5_3.xlsx` 读入整张表，手工切分为成本矩阵 `c`、需求向量 `d`（末行）、供应向量 `e`（末列）。目标与约束同上，用 `GLPK_MI` 求解后把解矩阵写入 `data/4_5_4.xlsx`。

In [ ]:
data = pd.read_excel("data/4_5_3.xlsx", header=None)
data = data.values
c = data[:-1, :-1]
d = data[-1, :-1]
e = data[:-1, -1]

x = cp.Variable((6, 8), pos=True)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)))
con = [cp.sum(x, axis=0) == d, cp.sum(x, axis=1) <= e]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
xd = pd.DataFrame(x.value)
xd.to_excel("data/4_5_4.xlsx")

## 2.9 分段值班人数优化（显式约束）

`x` 为 6 个整数变量（源程序 ex4_9_1.py），表示各时段安排的值班人数，目标最小化总人数 `sum(x)`。约束逐条列出相邻两个时段之和不低于该时段需求（35/40/50/45/55/30），并要求非负。用 `GLPK_MI` 求解并打印最优值与最优解。

In [ ]:
x = cp.Variable(6, integer=True)
obj = cp.Minimize(sum(x))
cons = [
    x[0] + x[5] >= 35,
    x[0] + x[1] >= 40,
    x[1] + x[2] >= 50,
    x[2] + x[3] >= 45,
    x[3] + x[4] >= 55,
    x[4] + x[5] >= 30,
    x >= 0,
]
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优值为：", prob.value)
print("最优解为：", x.value)

## 2.9 循环生成约束的等价写法

与上一单元同一问题的简洁重写（源程序 ex4_9_2.py）：把需求放入数组 `a`，用 `for i in range(6)` 配合 `x[(i-1)%6] + x[i] >= a[i]` 循环生成 6 条相邻时段覆盖约束（环形索引 `(i-1)%6` 实现首尾相接），目标与求解方式不变。

In [ ]:
a = np.array([35, 40, 50, 45, 55, 30])
x = cp.Variable(6, integer=True)
obj = cp.Minimize(sum(x))
cons = [x >= 0]
for i in range(6):
    cons.append(x[(i - 1) % 6] + x[i] >= a[i])
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优值为：", prob.value)
print("最优解为：", x.value)

## 2.10 0-1 指派问题

读取成本矩阵 `data/4_10.txt`（源程序 ex4_10.py）；`x` 为 $4\times 5$ 的 0-1 整数变量，目标最小化 `cp.sum(cp.multiply(c, x))`。约束：变量在 $[0,1]$、每列恰被指派一次（`cp.sum(x, axis=0)==1`）、每行至多指派 2 次（`cp.sum(x, axis=1)<=2`）。用 `GLPK_MI` 求解并打印最优解矩阵与最优值。

In [ ]:
c = np.loadtxt("data/4_10.txt")
x = cp.Variable((4, 5), integer=True)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)))
cons = [x >= 0, x <= 1, cp.sum(x, axis=0) == 1, cp.sum(x, axis=1) <= 2]
prob = cp.Problem(obj, cons)
prob.solve(solver="GLPK_MI")
print("最优解为：\n", x.value)
print("最优值为：", prob.value)

## 2.11 覆盖/选址型整数规划

读取 `data/4_11.txt`（源程序 ex4_11.py），先用两点欧氏距离 `np.linalg.norm` 构造 $10\times 10$ 距离矩阵 `d`。`x`（被选中的点）与 `y`（点对覆盖）均为 0-1 整数变量，目标最小化 `sum(x)`。约束含：至少覆盖一个点对 `sum(y)>=1`、每个点服务不超过 5 个邻居、以及 linking 约束 `x[i]==y[i,i]`、`d[i,j]*y[i,j]<=10*x[i]`（半径 10 内才可覆盖）、`x[i]>=y[i,j]`。用 `GLPK_MI` 求解并打印最优值、`x` 与 `y`。

In [ ]:
a = np.loadtxt("data/4_11.txt")
d = np.zeros((10, 10))
for i in range(10):
    for j in range(10):
        d[i, j] = np.linalg.norm(a[:, i] - a[:, j])
x = cp.Variable(10, integer=True)
y = cp.Variable((10, 10), integer=True)
obj = cp.Minimize(sum(x))
con = [sum(y) >= 1, cp.sum(y, axis=1) <= 5, x >= 0, x <= 1, y >= 0, y <= 1]
for i in range(10):
    con.append(x[i] == y[i, i])
    for j in range(10):
        con.append(d[i, j] * y[i, j] <= 10 * x[i])
        con.append(x[i] >= y[i, j])
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
print("----------\n", y.value)

## 2.12 投资组合收益随风险上限的变化

资产风险系数 `b`、收益系数 `c`、到期价值系数 `aeq`（源程序 ex4_12_1.py）。`x` 为 5 个非负投资额，目标最大化收益 `c @ x`；约束为投资总额折算后等于 $M=10000$（`aeq @ x == M`），且各风险资产的风险额 `b·x[1:]` 不超过风险上限 `a*M`。用 `while` 让风险上限 `a` 从 0 以 0.001 步长增至 0.05，逐次用 `GLPK_MI` 求解并记录 `(aa, Q)`，最后绘制收益 $Q$ 随 $a$ 变化的散点折线图。

In [ ]:
b = np.array([0.025, 0.015, 0.055, 0.026])
c = np.array([0.05, 0.27, 0.19, 0.185, 0.185])
x = cp.Variable(5, pos=True)
aeq = np.array([1, 1.01, 1.02, 1.045, 1.065])
obj = cp.Maximize(c @ x)
a = 0
aa = []
Q = []
X = []
M = 10000
while a < 0.05:
    con = [aeq @ x == M, cp.multiply(b, x[1:]) <= a * M]
    prob = cp.Problem(obj, con)
    prob.solve(solver="GLPK_MI")
    aa.append(a)
    Q.append(prob.value)
    X.append(x.value)
    a = a + 0.001
fig, ax = plt.subplots()
ax.plot(aa, Q, "r*")
ax.set(xlabel="$a$")
ax.set_ylabel("$Q$", rotation=0)
plt.show()

## 2.12 风险-收益双目标线性规划前沿

多目标组合模型的第二种写法（源程序 ex4_12_2.py）。6 个非负变量中前 5 个为各资产投资额、`x[5]` 为风险变量；`LP(w)` 对每个权重 `w[i]` 求解 `cp.Minimize(w[i]*x[5]-(1-w[i])*净收益)`，约束为投资总额=10000 与 `q·x[1:5] <= x[5]`，收集风险 `V=max(q*xx[:-1])`、收益 `Q`，绘制风险-收益折线。分别用等距权重 `w1=np.arange(0,1.1,0.1)` 与重点权重 `w2` 调用两次并打印结果。

In [ ]:
x = cp.Variable(6, pos=True)
r = np.array([0.05, 0.28, 0.21, 0.23, 0.25])
p = np.array([0, 0.01, 0.02, 0.045, 0.065])
q = np.array([0, 0.025, 0.015, 0.055, 0.026])


def LP(w):
    V = []
    Q = []
    X = []
    con = [(1 + p) @ x[:-1] == 10000, cp.multiply(q[1:], x[1:5]) <= x[5]]
    for i in range(len(w)):
        obj = cp.Minimize(w[i] * x[5] - (1 - w[i]) * ((r - p) @ x[:-1]))
        prob = cp.Problem(obj, con)
        prob.solve(solver="GLPK_MI")
        xx = x.value
        V.append(max(q * xx[:-1]))
        Q.append((r - p) @ xx[:-1])
        X.append(xx)
    print("w=", w)
    print("V=", np.round(V, 2))
    print("Q=", np.round(Q, 2))
    _fig, ax = plt.subplots()
    ax.plot(V, Q, "*-")
    ax.grid("on")
    ax.set(xlabel="风险（元）", ylabel="收益（元）")
    return X


w1 = np.arange(0, 1.1, 0.1)
LP(w1)
print("--------------")
w2 = np.array([0.766, 0.767, 0.810, 0.811, 0.824, 0.825, 0.962, 0.963, 1.0])
X = LP(w2)
print(X[-3])
plt.show()

## 2.13 旅行商问题（TSP）整数规划

读取关联矩阵 `data/4_13_1.xlsx`，将空值置 0（源程序 ex4_13.py）。按列构造加权邻接矩阵 `w`（$w_{ij}=\sum_k a_{ki}a_{kj}$，对角及虚拟点置 0），并存为 `data/4_13_2.xlsx`。以 `x`（0-1 边选择矩阵）与 `u`（MTZ 子序变量）建立 TSP：目标最小化 `cp.sum(cp.multiply(w, x))`，约束为每行每列度均为 1（构成哈密顿回路）以及 MTZ 消子回路约束 `u[i]-u[j]+(n+1)*x[i,j]<=n`。用 `GLPK_MI` 求解后打印最优值、解矩阵及所有 `x_ij=1` 的位置。

In [ ]:
a = pd.read_excel("data/4_13_1.xlsx", header=None)
a = a.values
a[np.isnan(a)] = 0
m, n = a.shape
w = np.ones((n + 1, n + 1)) * 10000000
for i in range(n):
    for j in range(n):
        if i != j:
            w[i, j] = sum(a[:, i] * a[:, j])
for i in range(n):
    w[i, n] = 0
    w[n, i] = 0
wd = pd.DataFrame(w)
wd.to_excel("data/4_13_2.xlsx")
x = cp.Variable((n + 1, n + 1), integer=True)
u = cp.Variable(n + 1, integer=True)
obj = cp.Minimize(cp.sum(cp.multiply(w, x)))
con = [
    cp.sum(x, axis=0) == 1,
    cp.sum(x, axis=1) == 1,
    x >= 0,
    x <= 1,
    u[0] == 0,
    u[1:] >= 1,
    u[1:] <= n,
]
for i in range(n + 1):
    for j in range(1, n + 1):
        con.append(u[i] - u[j] + (n + 1) * x[i, j] <= n)
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
i, j = np.nonzero(x.value)
print("xij=1 对应的行列位置如下：")
print(i + 1)
print(j + 1)